# Tutorial 01: Getting Started com hls4ml e PyTorch

Bem-vindo ao primeiro tutorial prático de aceleração de Machine Learning em FPGA!

### O que faremos neste tutorial:
1. **Compreender o problema físico**: Classificação de jatos de partículas subatômicas do CERN LHC (Jet Tagging).
2. **Treinar uma rede neural em PyTorch**: Arquitetura MLP de 3 camadas ocultas (`JetTagger`).
3. **Configurar o compilador hls4ml**: Definir a precisão de ponto fixo (`ap_fixed<16,6>`) e fator de reuso (`ReuseFactor = 1`).
4. **Gerar o código C++/HLS**: Inspecionar os arquivos de hardware gerados.
5. **Compilar e Simular em C++**: Usar o `g++` nativo para emulação de hardware com precisão bit a bit (*bit-accurate*).
6. **Avaliar os resultados**: Comparar acurácia e curvas ROC entre Float32 e Hardware.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# Adicionar diretório raiz para importar módulos auxiliares
sys.path.append('../..')
import models
import plotting
import hls4ml

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print(f"Versão do hls4ml: {hls4ml.__version__}")
print(f"Versão do PyTorch: {torch.__version__}")

## 1. Carregando os Dados de Teste
O dataset contém **16 variáveis de alto nível** calculadas pelos calorímetros do detector do CERN e **5 classes de jatos**:
- `g`: Glúon
- `q`: Quark leve
- `w`: Bóson W
- `z`: Bóson Z
- `t`: Quark Top

In [ ]:
data_dir = '../../data/jet-tagging'
X_train = np.load(os.path.join(data_dir, 'X_train.npy'))
y_train = np.load(os.path.join(data_dir, 'y_train.npy'))
X_test = np.load(os.path.join(data_dir, 'X_test.npy'))
y_test = np.load(os.path.join(data_dir, 'y_test.npy'))
classes = np.load(os.path.join(data_dir, 'classes.npy'), allow_pickle=True)

print(f"Shape de X_train: {X_train.shape}")
print(f"Shape de X_test:  {X_test.shape}")
print(f"Classes: {list(classes)}")

## 2. Definindo e Treinando a Rede Neural em PyTorch
A rede `JetTagger` possui:
- Entrada: 16 neurônios
- Camada Oculta 1: 64 neurônios + ReLU
- Camada Oculta 2: 32 neurônios + ReLU
- Camada Oculta 3: 32 neurônios + ReLU
- Saída: 5 neurônios + Softmax

In [ ]:
model = models.JetTagger()
weights_path = '../../models/pytorch_weights_part1.pt'

# Treinamento rápido de 5 épocas
n_samples = 50000
train_x = torch.tensor(X_train[:n_samples], dtype=torch.float32)
train_y = torch.tensor(np.argmax(y_train[:n_samples], axis=1), dtype=torch.long)
loader = DataLoader(TensorDataset(train_x, train_y), batch_size=256, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(1, 6):
    total_loss = 0.0
    for bx, by in loader:
        optimizer.zero_grad()
        out = model.logits(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Época {epoch}/5 | Loss: {total_loss / len(loader):.4f}")

torch.save(model.state_dict(), weights_path)
print(f"Pesos salvos em {weights_path}!")

# Avaliação Float32 no conjunto de teste
model.eval()
with torch.no_grad():
    test_x = torch.tensor(X_test, dtype=torch.float32)
    y_torch_pred = model(test_x).numpy()

acc_torch = accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_torch_pred, axis=1))
print(f"Acurácia do Modelo PyTorch (Float32): {acc_torch * 100:.2f}%")

## 3. Configurando o hls4ml para Síntese de Hardware
Criamos a configuração padrão com `config_from_pytorch_model`:
- **`Precision`**: `ap_fixed<16,6>` (16 bits no total, 6 inteiros e 10 fracionários).
- **`ReuseFactor`**: `1` (paralelismo total).

In [ ]:
config = hls4ml.utils.config_from_pytorch_model(
    model,
    input_shape=(16,),
    granularity='model',
    backend='Vivado',
)

print("Configuração de Hardware:")
plotting.print_dict(config)

## 4. Convertendo o Modelo PyTorch para C++/HLS
O conversor gera toda a estrutura de firmware em C++ na pasta `hls4ml_prj_base_part1`.

In [ ]:
hls_output_dir = 'hls4ml_prj_base_part1'
hls_model = hls4ml.converters.convert_from_pytorch_model(
    model,
    hls_config=config,
    output_dir=hls_output_dir,
    part='xcu200-fsgd2104-2-e',
)
hls_model.write()
print(f"Projeto de firmware gerado com sucesso em: {hls_output_dir}!")

## 5. Compilando com g++ e Executando Simulação em Ponto Fixo
A chamada `hls_model.compile()` invoca o compilador `g++` para gerar uma biblioteca compartilhada `.so`.
Com ela, podemos alimentar dados e simular a precisão exata do hardware sem precisar de uma placa física.

In [ ]:
hls_model.compile()

# Avaliar nas primeiras 5000 amostras
eval_n = min(5000, len(X_test))
y_hls_pred = hls_model.predict(X_test[:eval_n])

acc_hls = accuracy_score(np.argmax(y_test[:eval_n], axis=1), np.argmax(y_hls_pred, axis=1))
acc_torch_eval = accuracy_score(np.argmax(y_test[:eval_n], axis=1), np.argmax(y_torch_pred[:eval_n], axis=1))

print(f"Acurácia PyTorch (Float32) : {acc_torch_eval * 100:.2f}%")
print(f"Acurácia hls4ml (ap_fixed)  : {acc_hls * 100:.2f}%")
print(f"Diferença de Precisão       : {abs(acc_torch_eval - acc_hls) * 100:.2f}%")

## 6. Comparação Visual: Curvas ROC
Traçamos a curva ROC de cada classe comparando a linha sólida (PyTorch Float32) com a linha tracejada (hls4ml Ponto Fixo).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
plotting.makeRoc(y_test[:eval_n], y_torch_pred[:eval_n], list(classes), linestyle='-')
plt.gca().set_prop_cycle(None)
plotting.makeRoc(y_test[:eval_n], y_hls_pred, list(classes), linestyle='--')

from matplotlib.lines import Line2D
lines = [
    Line2D([0], [0], color='black', linestyle='-', label='PyTorch (Float32)'),
    Line2D([0], [0], color='black', linestyle='--', label='hls4ml (ap_fixed<16,6>)')
]
ax.legend(handles=lines, loc='lower right')
plt.tight_layout()
plt.show()